# A1 · Gradient boosting for regression from scratch: trees on the residuals

**Type**: Implement from scratch · **Level**: ●●○ independent · **Time**: about 50 minutes · **core path ★**

## Objective

Implement gradient boosting with the squared loss using only `DecisionTreeRegressor` as a building block: constant initial prediction, small trees on the residuals, update with shrinkage. Check that it matches `GradientBoostingRegressor` in the first rounds, measure the error curve as a function of the
rounds and find the round at which the test error starts to get worse.

## Dataset

`generate()` (given): 600 points with 3 uniform features in [−3, 3] and `y = sin(x0) + 0.09 x1² + noise(0.3)`, 60/40 split: `X_tr, y_tr, X_te, y_te`.

## Educational purpose

Understand that boosting is a four-line loop: residuals → small tree → update with step `eta`. See the two central parameters (`eta` and the number of rounds) in action and the difference from the forest: here the test error **rises** if you go on too long.

## Task

1. `fit_gb(X, y, n_rounds, eta, depth)`: `f0 = y.mean()`; for each round fit `DecisionTreeRegressor(max_depth=depth, criterion="squared_error", random_state=0)` to the residuals `y − F` and update `F += eta * tree.predict(X)`. Return `(f0, trees)`.
2. `predict_gb(model, X, eta)` and `error_curve(model, X, y, eta)`: the MSE after each round (list of `n_rounds` values; the partial predictions are built by adding one tree at a time).
3. Train `model = fit_gb(X_tr, y_tr, 300, 0.3, 2)` and save `train_curve` and `test_curve` (MSE per round), `best_round` (1-based index of the minimum of `test_curve`) and `best_mse`.
4. Compare with `GradientBoostingRegressor(n_estimators=..., learning_rate=0.1, max_depth=3, criterion="squared_error", random_state=0)`: save `diff_5_rounds` (maximum absolute difference between the test predictions after **5 rounds** with `eta=0.1, depth=3`) and `diff_100_rounds` (after 100 rounds: small differences accumulate because of the tie-breaking and the 32-bit precision of scikit-learn).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor


def generate():
    rng = np.random.default_rng(0)
    X = rng.uniform(-3, 3, (600, 3))
    y = np.sin(X[:, 0]) + 0.09 * X[:, 1] ** 2 + rng.normal(0, 0.3, 600)
    return train_test_split(X, y, test_size=0.4, random_state=0)


X_tr, X_te, y_tr, y_te = generate()

## Your work

In [ ]:
def fit_gb(X, y, n_rounds, eta, depth):
    ...


def predict_gb(model, X, eta):
    ...


def error_curve(model, X, y, eta):
    ...

In [ ]:
model = fit_gb(X_tr, y_tr, 300, 0.3, 2)
train_curve = None
test_curve = None
best_round = None
best_mse = None

In [ ]:
diff_5_rounds = None
diff_100_rounds = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Loop: `F = np.full(len(y), y.mean()); for _ in range(n_rounds): t = DecisionTreeRegressor(...).fit(X, y - F); F += eta * t.predict(X); trees.append(t)`. Curve on the test set: `F = f0; for t in trees: F = F + eta * t.predict(X_te); curve.append(mean((y_te - F) ** 2))`. Comparison: `GradientBoostingRegressor(n_estimators=5, learning_rate=0.1, max_depth=3, criterion="squared_error", random_state=0).fit(X_tr, y_tr).predict(X_te)`.'):
    check(len(model[1]) == 300 and close(model[0], y_tr.mean()), "300 trees and initial prediction equal to the mean", "f0 = y.mean()")
    check(len(train_curve) == 300 and all(np.diff(train_curve) <= 1e-9), "the training error never goes up", "each tree reduces (or leaves equal) the training error")
    check(close(test_curve[-1], np.mean((predict_gb(model, X_te, 0.3) - y_te) ** 2)) and close(train_curve[0], np.mean((y_tr - (y_tr.mean() + 0.3 * model[1][0].predict(X_tr))) ** 2)), "curves consistent with the full predictions", "MSE after each round")
    check(train_curve[-1] < 0.05 and test_curve[-1] > test_curve[best_round - 1] + 0.02 and 5 <= best_round <= 100, "the training error goes below 0.05; the test error has its minimum between round 5 and 100 and then climbs back by over 0.02", "overfitting: with eta=0.3 the test error climbs back")
    check(close(best_mse, min(test_curve)) and best_mse < 0.15, "minimum test MSE below 0.15", "min(test_curve)")
    check(diff_5_rounds < 1e-9 and diff_100_rounds < 0.25, "after 5 rounds the predictions match scikit-learn (1e-9); after 100 they differ by at most 0.25", "criterion='squared_error', random_state=0")
done()

---
The commented solution is in `solutions/A_from_scratch/A1_regression_gradient_boosting_from_scratch.ipynb`. Open it after you pass the check, or after you have really tried.